# Predicting Gender Norm Internalization: WNYC Masculinity Survey

This notebook builds binary classifiers on two attitudinal targets from the WNYC/FiveThirtyEight masculinity survey:

- **Primary — Q17**: *Do you feel expected to make the first move in romantic relationships?* (Yes/No)  
  A concrete, behaviorally-grounded measure of **internalized traditional courtship norms**.

- **Secondary — Q5**: *Does society put unhealthy pressure on men?* (Yes/No)  
  A social/attitudinal belief about **structural gender pressure**.

Both targets are predicted from lifestyle behaviors, workplace attitudes, relationship patterns, worry profiles, and demographics — with no use of self-reported masculinity ratings (Q1/Q2), which are circular.

**Data sources:**  
- `raw-responses.csv` — original string-valued survey responses  
- `cleaned-responses.csv` — preprocessed binary/ordinal encodings from original analysis

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

raw = pd.read_csv('raw-responses.csv')
df_clean = pd.read_csv('cleaned-responses.csv')

print(f"Raw responses: {raw.shape}")
print(f"Cleaned responses: {df_clean.shape}")
raw.head(3)

## 2. Target Variable Exploration

In [ ]:
# Q17: Expected to make first move
q17_counts = raw['q0017'].value_counts()
# Q5: Society puts unhealthy pressure on men
q5_counts = raw['q0005'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors_17 = ['#2196F3', '#FF5722', '#9E9E9E']
axes[0].bar(q17_counts.index, q17_counts.values, color=colors_17[:len(q17_counts)])
axes[0].set_title('Q17: Expected to make the first move?', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(q17_counts.items()):
    axes[0].text(i, val + 10, f'{val}\n({val/len(raw)*100:.1f}%)', ha='center', fontsize=10)

colors_5 = ['#4CAF50', '#F44336', '#9E9E9E']
axes[1].bar(q5_counts.index, q5_counts.values, color=colors_5[:len(q5_counts)])
axes[1].set_title('Q5: Society puts unhealthy pressure on men?', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
for i, (idx, val) in enumerate(q5_counts.items()):
    axes[1].text(i, val + 10, f'{val}\n({val/len(raw)*100:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.suptitle('Target Variable Distributions', fontsize=15, y=1.02, fontweight='bold')
plt.show()

print("Q17 — Expected to make first move:")
print(q17_counts.to_string())
print("\nQ5 — Society puts unhealthy pressure:")
print(q5_counts.to_string())

## 3. Feature Engineering

Features are drawn from 9 question blocks:

| Block | Question | Type | Notes |
|---|---|---|---|
| Q4 | Source of masculinity ideas | multi-select binary | father, mother, family, pop culture, friends, other |
| Q7 | Lifestyle frequency | ordinal 0–4 | Often→4, Never not open→0 |
| Q8 | Daily worries | multi-select binary | 12 worry items |
| Q9 | Employment status | binary | employed/not |
| Q10/Q11 | Work advantages/disadvantages | multi-select binary | conditional on employed |
| Q12 | Harassment response | multi-select binary | conditional on employed |
| Q14 | Heard about #MeToo | ordinal | conditional on employed |
| Q18 | Tries to pay on dates | ordinal 1–5 | Never→1, Always→5 |
| Q19–Q21 | Consent & relationship behaviors | multi-select binary | |
| Q22 | Changed behavior post-#MeToo | binary | |
| Demographics | Marital, orientation, race, education, age, kids | various | |

In [ ]:
def build_features(raw, df_clean, exclude_col=None):
    """Assemble the full feature matrix from raw and cleaned data.
    
    exclude_col: column name to drop (use when that column IS the target).
    """
    # Q4 — source of masculinity ideas (binary flags from cleaned)
    q4_cols = [c for c in df_clean.columns if c.startswith('q0004')]

    # Q7 — lifestyle frequency matrix (ordinal-encode from raw)
    q7_map = {
        'Often': 4, 'Sometimes': 3, 'Rarely': 2,
        'Never, but open to it': 1, 'Never, and not open to it': 0,
        'No answer': np.nan
    }
    q7_cols = [c for c in raw.columns if c.startswith('q0007')]
    q7_df = raw[q7_cols].replace(q7_map)

    # Q8 — daily worries (binary from cleaned)
    q8_cols = [c for c in df_clean.columns if c.startswith('q0008')]

    # Q9 — employment
    q9 = df_clean['q0009'].map({'employed': 1, 'not_employed': 0})

    # Q10/Q11 — work advantages/disadvantages (binary from cleaned)
    q10_cols = [c for c in df_clean.columns if c.startswith('q0010')]
    q11_cols = [c for c in df_clean.columns if c.startswith('q0011')]

    # Q12 — harassment response (binary from cleaned)
    q12_cols = [c for c in df_clean.columns if c.startswith('q0012')]

    # Q14 — heard about #MeToo (ordinal; 0 = not employed, treat as missing)
    q14 = df_clean['auto_q0014'].replace({0: np.nan})

    # Q17 — expected to make first move (binary; may be target or feature)
    q17 = raw['q0017'].map({'Yes': 1, 'No': 0})

    # Q18 — tries to pay on dates (ordinal)
    q18_map = {'Always': 5, 'Often': 4, 'Sometimes': 3, 'Rarely': 2, 'Never': 1, 'No answer': np.nan}
    q18 = raw['q0018'].map(q18_map)

    # Q19–Q21 — consent/relationship behaviors (binary from cleaned)
    q19_cols = [c for c in df_clean.columns if c.startswith('q0019')]
    q20_cols = [c for c in df_clean.columns if c.startswith('q0020')]
    q21_cols = [c for c in df_clean.columns if c.startswith('q0021')]

    # Q22 — changed behavior post-#MeToo
    q22 = df_clean['auto_q0022'].map({0: 0, 1: 1, 2: np.nan})

    # Demographics
    marital     = df_clean['auto_q0024']
    orientation = df_clean['auto_orientation']
    race        = df_clean['auto_race2']
    educ        = df_clean['auto_educ4']
    age         = df_clean['auto_age3']
    kids        = df_clean['auto_kids'].map({0: 0, 1: 1, 2: np.nan})

    feat = pd.concat([
        df_clean[q4_cols],
        q7_df.reset_index(drop=True),
        df_clean[q8_cols],
        q9.rename('q0009'),
        df_clean[q10_cols], df_clean[q11_cols], df_clean[q12_cols],
        q14.rename('q0014'),
        q17.reset_index(drop=True).rename('q0017'),
        q18.reset_index(drop=True).rename('q0018'),
        df_clean[q19_cols], df_clean[q20_cols], df_clean[q21_cols],
        q22.rename('q0022'),
        marital.rename('marital'),
        orientation.rename('orientation'),
        race.rename('race'),
        educ.rename('educ'),
        age.rename('age'),
        kids.rename('kids'),
    ], axis=1)

    if exclude_col and exclude_col in feat.columns:
        feat = feat.drop(columns=[exclude_col])

    return feat


# --- Q17 dataset (exclude q0017 from features) ---
feat17 = build_features(raw, df_clean, exclude_col='q0017')
y17 = raw['q0017'].map({'Yes': 1, 'No': 0}).reset_index(drop=True)
mask17 = y17.notna()
X17_raw = feat17[mask17].copy()
y17 = y17[mask17].copy()

imp17 = SimpleImputer(strategy='median')
X17 = pd.DataFrame(imp17.fit_transform(X17_raw), columns=X17_raw.columns)

# --- Q5 dataset (q0017 IS a feature) ---
feat05 = build_features(raw, df_clean, exclude_col=None)
y05 = raw['q0005'].map({'Yes': 1, 'No': 0}).reset_index(drop=True)
mask05 = y05.notna()
X05_raw = feat05[mask05].copy()
y05 = y05[mask05].copy()

imp05 = SimpleImputer(strategy='median')
X05 = pd.DataFrame(imp05.fit_transform(X05_raw), columns=X05_raw.columns)

print(f"Q17 dataset: {X17.shape[0]} rows, {X17.shape[1]} features")
print(f"  Class balance — Yes: {(y17==1).sum()}, No: {(y17==0).sum()}")
print(f"\nQ5 dataset:  {X05.shape[0]} rows, {X05.shape[1]} features")
print(f"  Class balance — Yes: {(y05==1).sum()}, No: {(y05==0).sum()}")

## 4. EDA: Feature Distributions by Target

In [ ]:
# Q7 lifestyle frequencies split by Q17 target
q7_labels = {
    'q0007_0001': 'Ask friend\n(professional advice)',
    'q0007_0002': 'Ask friend\n(personal advice)',
    'q0007_0003': 'Physical affection\n(male friends)',
    'q0007_0004': 'Cry',
    'q0007_0005': 'Sex with women',
    'q0007_0006': 'Sex with men',
    'q0007_0007': 'Watch sports',
    'q0007_0008': 'Work out',
    'q0007_0009': 'See therapist',
    'q0007_0010': 'See therapist (2)',
    'q0007_0011': 'Feel lonely/isolated',
}

q7_map = {'Often': 4, 'Sometimes': 3, 'Rarely': 2,
          'Never, but open to it': 1, 'Never, and not open to it': 0, 'No answer': np.nan}
q7_cols = [c for c in raw.columns if c.startswith('q0007')]
q7_df = raw[q7_cols].replace(q7_map)

plot_df = q7_df.copy()
plot_df['target'] = raw['q0017'].map({'Yes': 'Yes — feels expected', 'No': 'No — does not feel expected'})
plot_df = plot_df.dropna(subset=['target'])

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(q7_cols):
    ax = axes[i]
    means = plot_df.groupby('target')[col].mean()
    colors = ['#2196F3', '#FF5722']
    bars = ax.bar(range(len(means)), means.values, color=colors, width=0.5)
    ax.set_title(q7_labels.get(col, col), fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(['Yes', 'No'], fontsize=8)
    ax.set_ylabel('Mean frequency\n(0=Never, 4=Often)', fontsize=7)
    ax.set_ylim(0, 4.5)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.05, f'{val:.2f}', 
                ha='center', va='bottom', fontsize=8)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Q7: Lifestyle Frequencies by Q17 Target (Expected to Make First Move?)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Q8 worry rates split by Q17
q8_labels = {
    'q0008_0001': 'Height',
    'q0008_0002': 'Weight',
    'q0008_0003': 'Hair/hairline',
    'q0008_0004': 'Physique',
    'q0008_0005': 'Genitalia appearance',
    'q0008_0006': 'Clothing/style',
    'q0008_0007': 'Sexual performance',
    'q0008_0008': 'Mental health',
    'q0008_0009': 'Physical health',
    'q0008_0010': 'Finances/income',
    'q0008_0011': 'Ability to provide',
    'q0008_0012': 'None of above',
}

q8_cols = [c for c in df_clean.columns if c.startswith('q0008')]
q8_plot = df_clean[q8_cols].copy()
q8_plot['target'] = raw['q0017'].map({'Yes': 'Yes', 'No': 'No'}).values
q8_plot = q8_plot.dropna(subset=['target'])

yes_rates = q8_plot[q8_plot['target'] == 'Yes'][q8_cols].mean()
no_rates  = q8_plot[q8_plot['target'] == 'No'][q8_cols].mean()
diff = yes_rates - no_rates

diff_df = pd.DataFrame({
    'label': [q8_labels[c] for c in q8_cols],
    'diff': diff.values
}).sort_values('diff')

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2196F3' if d > 0 else '#FF5722' for d in diff_df['diff']]
ax.barh(diff_df['label'], diff_df['diff'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Difference in worry rate (Yes − No)', fontsize=11)
ax.set_title('Q8: Daily Worries — Rate Difference by Q17\n(Blue = more common among those who feel expected to initiate)',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

In [ ]:
# Q18 (pays on dates) distribution by Q17
q18_map = {'Always': 5, 'Often': 4, 'Sometimes': 3, 'Rarely': 2, 'Never': 1, 'No answer': np.nan}
q18_labels = {5: 'Always', 4: 'Often', 3: 'Sometimes', 2: 'Rarely', 1: 'Never'}

q18_plot = pd.DataFrame({
    'q0018': raw['q0018'].map(q18_map),
    'q0017': raw['q0017']
}).dropna()

ct = q18_plot.groupby(['q0017', 'q0018']).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0)
ct_pct.columns = [q18_labels[c] for c in ct_pct.columns]

ax = ct_pct.T.plot(kind='bar', figsize=(9, 5), color=['#2196F3', '#FF5722'], 
                   width=0.6, edgecolor='white')
ax.set_xlabel('How often tries to pay on dates', fontsize=11)
ax.set_ylabel('Proportion', fontsize=11)
ax.set_title('Q18: Pays on Dates — by Q17 (Feels Expected to Make First Move?)',
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(title='Q17', labels=['No', 'Yes'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Modeling — Q17 (Primary Target)

Three classifiers evaluated with 5-fold stratified cross-validation:
- **Logistic Regression** (L2, C=1.0) — most interpretable
- **Random Forest** (200 trees) — handles nonlinearity, good for feature importance
- **Gradient Boosting** (200 estimators, lr=0.05) — often best AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_17 = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, C=1.0))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                     max_depth=4, random_state=42),
}

results_17 = {}
majority_baseline = (y17 == y17.mode()[0]).mean()

print(f"Majority class baseline accuracy: {majority_baseline:.3f}\n")
print(f"{'Model':<25} {'Accuracy':>10} {'AUC':>10}")
print('-' * 47)

for name, model in models_17.items():
    acc = cross_val_score(model, X17, y17, cv=cv, scoring='accuracy')
    auc = cross_val_score(model, X17, y17, cv=cv, scoring='roc_auc')
    results_17[name] = {'acc_mean': acc.mean(), 'acc_std': acc.std(),
                         'auc_mean': auc.mean(), 'auc_std': auc.std()}
    print(f"{name:<25} {acc.mean():.3f}±{acc.std():.3f}  {auc.mean():.3f}±{auc.std():.3f}")

In [ ]:
# ROC curves for Q17
X17_train, X17_test, y17_train, y17_test = train_test_split(
    X17, y17, test_size=0.2, random_state=42, stratify=y17
)

fig, ax = plt.subplots(figsize=(7, 6))

colors_roc = ['#1976D2', '#388E3C', '#F57C00']
for (name, model), color in zip(models_17.items(), colors_roc):
    model.fit(X17_train, y17_train)
    RocCurveDisplay.from_estimator(model, X17_test, y17_test, ax=ax,
                                   name=name, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random baseline')
ax.set_title('ROC Curves — Q17: Expected to Make First Move?',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Classification report for best model (Random Forest)
rf17 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf17.fit(X17_train, y17_train)
y17_pred = rf17.predict(X17_test)

print("Random Forest — Q17 Classification Report:")
print(classification_report(y17_test, y17_pred, target_names=['No (0)', 'Yes (1)']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y17_test, y17_pred,
    display_labels=['Not expected', 'Expected'], ax=ax,
    colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Q17 (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature Importance — Q17

In [ ]:
# Permutation importance
perm17 = permutation_importance(rf17, X17_test, y17_test,
                                 n_repeats=15, random_state=42, n_jobs=-1)
fi17 = pd.DataFrame({
    'feature': X17.columns,
    'importance': perm17.importances_mean,
    'std': perm17.importances_std
}).sort_values('importance', ascending=False).head(20)

# Human-readable labels
feat_labels = {
    'q0018':      'Q18: Tries to pay on dates',
    'q0007_0005': 'Q7: Sex with women (freq)',
    'q0007_0006': 'Q7: Sex with men (freq)',
    'q0007_0011': 'Q7: Feels lonely/isolated (freq)',
    'q0007_0003': 'Q7: Physical affection — male friends (freq)',
    'q0007_0001': 'Q7: Asks friend for professional advice',
    'q0007_0008': 'Q7: Works out (freq)',
    'q0007_0009': 'Q7: Sees therapist (freq)',
    'q0004_0001': 'Q4: Masculinity ideas ← father',
    'q0004_0005': 'Q4: Masculinity ideas ← friends',
    'q0008_0010': 'Q8: Worries about finances',
    'q0008_0008': 'Q8: Worries about mental health',
    'q0008_0002': 'Q8: Worries about weight',
    'q0008_0005': 'Q8: Worries about genitalia appearance',
    'q0011_0001': 'Q11: Disadvantage — managers prefer women',
    'q0011_0002': 'Q11: Disadvantage — harassment risk',
    'q0011_0003': 'Q11: Disadvantage — sexism/racism accusation risk',
    'q0010_0001': 'Q10: Advantage — men earn more',
    'q0010_0003': 'Q10: Advantage — men have more choice',
    'q0010_0005': 'Q10: Advantage — men praised more',
    'q0012_0005': 'Q12: Harassment response — did nothing',
    'q0021_0001': 'Q21: Wondered if pushed partner too far',
    'q0021_0003': 'Q21: Contacted past partner re: consent',
    'q0019_0005': 'Q19: Pays — because asked them out',
    'q0019_0002': 'Q19: Pays — earns more than date',
    'q0014':      'Q14: Heard about #MeToo',
    'q0022':      'Q22: Changed behavior post-#MeToo',
    'orientation':'Demographics: Sexual orientation',
    'race':       'Demographics: Race (white/non-white)',
    'educ':       'Demographics: Education level',
    'age':        'Demographics: Age group',
    'kids':       'Demographics: Has children',
    'marital':    'Demographics: Marital status',
    'q0009':      'Q9: Employment status',
}

fi17['label'] = fi17['feature'].map(feat_labels).fillna(fi17['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#1976D2' if v > 0 else '#E0E0E0' for v in fi17['importance']]
bars = ax.barh(fi17['label'][::-1], fi17['importance'][::-1], 
               xerr=fi17['std'][::-1], color=colors[::-1],
               capsize=3, edgecolor='white')
ax.set_xlabel('Permutation Importance (mean accuracy drop)', fontsize=11)
ax.set_title('Top 20 Features — Q17: Expected to Make First Move?\n(Random Forest, permutation importance)',
             fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression coefficients — directional effect
scaler17 = StandardScaler()
lr17 = LogisticRegression(max_iter=1000, C=0.1)
lr17.fit(scaler17.fit_transform(X17_train), y17_train)

lr_coef17 = pd.DataFrame({
    'feature': X17.columns,
    'coef': lr17.coef_[0]
}).sort_values('coef', key=abs, ascending=False).head(20)
lr_coef17['label'] = lr_coef17['feature'].map(feat_labels).fillna(lr_coef17['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#1976D2' if c > 0 else '#FF5722' for c in lr_coef17['coef']]
ax.barh(lr_coef17['label'][::-1], lr_coef17['coef'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient', fontsize=11)
ax.set_title('LR Coefficients — Q17 (Blue = predicts "Yes", Red = predicts "No")',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Blue (positive) = associated with feeling expected to make the first move")
print("  Red (negative)  = associated with NOT feeling expected to initiate")

## 7. Modeling — Q5 (Secondary Target)

**Q5**: Does society put unhealthy or bad pressure on men? (Yes=1, No=0)  
Here Q17 is included as a feature — feeling expected to initiate is itself a signal about perceived social pressure.

In [ ]:
models_05 = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, C=1.0))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                     max_depth=4, random_state=42),
}

majority_baseline_05 = (y05 == y05.mode()[0]).mean()
print(f"Majority class baseline accuracy: {majority_baseline_05:.3f}\n")
print(f"{'Model':<25} {'Accuracy':>10} {'AUC':>10}")
print('-' * 47)

results_05 = {}
for name, model in models_05.items():
    acc = cross_val_score(model, X05, y05, cv=cv, scoring='accuracy')
    auc = cross_val_score(model, X05, y05, cv=cv, scoring='roc_auc')
    results_05[name] = {'acc_mean': acc.mean(), 'acc_std': acc.std(),
                         'auc_mean': auc.mean(), 'auc_std': auc.std()}
    print(f"{name:<25} {acc.mean():.3f}±{acc.std():.3f}  {auc.mean():.3f}±{auc.std():.3f}")

In [ ]:
# ROC curves for Q5
X05_train, X05_test, y05_train, y05_test = train_test_split(
    X05, y05, test_size=0.2, random_state=42, stratify=y05
)

fig, ax = plt.subplots(figsize=(7, 6))
for (name, model), color in zip(models_05.items(), colors_roc):
    model.fit(X05_train, y05_train)
    RocCurveDisplay.from_estimator(model, X05_test, y05_test, ax=ax,
                                   name=name, color=color)
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random baseline')
ax.set_title('ROC Curves — Q5: Society Puts Unhealthy Pressure on Men?',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — Q5
rf05 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf05.fit(X05_train, y05_train)

perm05 = permutation_importance(rf05, X05_test, y05_test,
                                 n_repeats=15, random_state=42, n_jobs=-1)
fi05 = pd.DataFrame({
    'feature': X05.columns,
    'importance': perm05.importances_mean,
    'std': perm05.importances_std
}).sort_values('importance', ascending=False).head(20)
fi05['label'] = fi05['feature'].map(feat_labels).fillna(fi05['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors05 = ['#388E3C' if v > 0 else '#E0E0E0' for v in fi05['importance']]
ax.barh(fi05['label'][::-1], fi05['importance'][::-1],
        xerr=fi05['std'][::-1], color=colors05[::-1],
        capsize=3, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlabel('Permutation Importance (mean accuracy drop)', fontsize=11)
ax.set_title('Top 20 Features — Q5: Society Puts Unhealthy Pressure on Men?\n(Random Forest, permutation importance)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# LR coefficients — Q5
scaler05 = StandardScaler()
lr05 = LogisticRegression(max_iter=1000, C=0.1)
lr05.fit(scaler05.fit_transform(X05_train), y05_train)

lr_coef05 = pd.DataFrame({
    'feature': X05.columns,
    'coef': lr05.coef_[0]
}).sort_values('coef', key=abs, ascending=False).head(20)
lr_coef05['label'] = lr_coef05['feature'].map(feat_labels).fillna(lr_coef05['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors05_lr = ['#388E3C' if c > 0 else '#FF5722' for c in lr_coef05['coef']]
ax.barh(lr_coef05['label'][::-1], lr_coef05['coef'][::-1],
        color=colors05_lr[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient', fontsize=11)
ax.set_title('LR Coefficients — Q5 (Green = predicts "Yes", Red = predicts "No")',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Head-to-Head Comparison

In [ ]:
model_names = list(results_17.keys())
x = np.arange(len(model_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax = axes[0]
bars1 = ax.bar(x - width/2, [results_17[m]['acc_mean'] for m in model_names],
               width, label='Q17 (first move)', color='#1976D2', alpha=0.85)
bars2 = ax.bar(x + width/2, [results_05[m]['acc_mean'] for m in model_names],
               width, label='Q5 (society pressure)', color='#388E3C', alpha=0.85)
ax.axhline(majority_baseline, color='#1976D2', linestyle='--', lw=1, alpha=0.5, label=f'Q17 baseline ({majority_baseline:.2f})')
ax.axhline(majority_baseline_05, color='#388E3C', linestyle='--', lw=1, alpha=0.5, label=f'Q5 baseline ({majority_baseline_05:.2f})')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10)
ax.set_ylabel('Cross-validated Accuracy')
ax.set_title('Accuracy Comparison', fontweight='bold')
ax.set_ylim(0.55, 0.75)
ax.legend(fontsize=8)

# AUC
ax = axes[1]
ax.bar(x - width/2, [results_17[m]['auc_mean'] for m in model_names],
       width, label='Q17 (first move)', color='#1976D2', alpha=0.85)
ax.bar(x + width/2, [results_05[m]['auc_mean'] for m in model_names],
       width, label='Q5 (society pressure)', color='#388E3C', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', lw=1, label='Random baseline (0.5)')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10)
ax.set_ylabel('Cross-validated AUC')
ax.set_title('AUC Comparison', fontweight='bold')
ax.set_ylim(0.45, 0.75)
ax.legend(fontsize=8)

plt.suptitle('Q17 vs Q5: Model Performance Comparison (5-fold CV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  Q17 best AUC: {max(v['auc_mean'] for v in results_17.values()):.3f} (RF)")
print(f"  Q5  best AUC: {max(v['auc_mean'] for v in results_05.values()):.3f} (LR)")
print("\n  Q17 is the more predictable target across all models.")
print("  Q5 signal is weaker — the survey's behavioral features capture enacted")
print("  norms better than social/political beliefs.")

## 9. Key Findings

### Q17 — Expected to make the first move

**Predicts YES (feels expected to initiate):**
- **Tries to pay on dates** (strongest signal) — men who always pay also feel expected to initiate; both reflect the same traditional courtship script
- **Feels lonely/isolated frequently** — possibly internalizing pressure to be the active pursuer
- **Learned masculinity norms from father** — direct transmission of traditional role expectations
- **Perceives men earn more at work** — provider-role mindset extends to romantic context
- **Has kids, white race** — demographic markers for traditional norm adoption
- **Didn't respond to witnessed harassment** — norm-passive, not norm-questioning

**Predicts NO (does not feel expected to initiate):**
- **Contacted past partner to check on consent** — active reflection on and pushback against traditional scripts
- **Higher education** — associated with questioning of inherited norms
- **Non-straight orientation** — outside the heteronormative courtship script
- **Perceives men are explicitly praised more** — men who notice gender asymmetries at work are less likely to internalize them romantically

### Q5 — Society puts unhealthy pressure on men

**Predicts YES:**
- **Feels lonely/isolated frequently** — emotional burden as evidence of systemic pressure
- **Sees a therapist** — self-aware about the cost of masculine expectations
- **Worries about finances and sexual performance** — experiencing the specific pressures the question asks about
- **Tries to pay on dates** — enacting the script while also recognizing its burden

**Predicts NO:**
- **Non-straight orientation** — less subject to heteronormative masculine pressure
- **Works out frequently** — comfortable with / performing traditional masculinity
- **Never witnessed harassment** — generally non-reflective about gender dynamics
- **Got masculinity ideas from other family members** (not father/pop culture) — more diffuse, less scripted source

### Performance ceiling

The ~0.65–0.67 AUC ceiling is likely real, not a modeling artifact. Attitudinal survey responses are genuinely noisy — people hold contradictory beliefs, and the behavioral features in this survey capture tendencies, not deterministic rules. Further tuning (polynomial features, XGBoost, etc.) is unlikely to meaningfully improve AUC without introducing leakage.